In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

In [ ]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from xgboost import XGBRegressor

In [ ]:
df=pd.read_csv("Steel_industry_data.csv")
print(df.shape)
df.head()

(35040, 11)


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [ ]:
df['date']=pd.to_datetime(df['date'],
                          dayfirst=True)
df['hour']=df['date'].dt.hour
df['month']=df['date'].dt.month
df['is_weekend']=(df['WeekStatus']==
                  "Weekend").astype(int)
labelenco_day_of_week=LabelEncoder()
labelenco_load=LabelEncoder()
df['Day_of_week_encoded']=labelenco_day_of_week.fit_transform(df['Day_of_week'])
df['Load_type_encoded']=labelenco_load.fit_transform(df['Load_Type'])
joblib.dump(labelenco_day_of_week,"encoder_day.pkl")
joblib.dump(labelenco_load,"encoder_load.pkl")

cols_to_drop = ['date','WeekStatus','Day_of_week',"Load_Type",'CO2(tCO2)']
df.drop(columns=cols_to_drop,inplace=True)



In [ ]:
Y="Usage_kWh"
X=df[[col for col in df.columns if col!=Y]]

X_train,X_test,Y_train,Y_test=train_test_split(X,df[Y],test_size=0.2,random_state=42)
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
joblib.dump(scaler,"scaler.pkl")

['scaler.pkl']

In [ ]:
joblib.dump(list(X.columns),"feature_columns.pkl")

['feature_columns.pkl']

In [ ]:
# model 1: linear regression
model1 = LinearRegression()
model1.fit(X_train_scaled, Y_train)
prediction1=model1.predict(X_test_scaled)
mae1=mean_absolute_error(Y_test,prediction1)
rmse1=np.sqrt(mean_squared_error(Y_test,prediction1))
r2_score1=r2_score(Y_test,prediction1)

In [ ]:
joblib.dump(model1,"model1.pkl")

['model1.pkl']

In [ ]:
#model 2: random forest
model2=RandomForestRegressor(random_state=42)
model2.fit(X_train,Y_train)
prediction2=model2.predict(X_test)
mae2=mean_absolute_error(Y_test,prediction2)
rmse2=np.sqrt(mean_squared_error(Y_test,prediction2))
r2_score2=r2_score(Y_test,prediction2)

In [ ]:
joblib.dump(model2,"model2.pkl")

['model2.pkl']

In [ ]:
#model 3: xg boost
model3=XGBRegressor(random_state=42)
model3.fit(X_train,Y_train)
prediction3=model3.predict(X_test)
mae3=mean_absolute_error(Y_test,prediction3)
rmse3=np.sqrt(mean_squared_error(Y_test,prediction3))
r2_score3=r2_score(Y_test,prediction3)

In [ ]:
joblib.dump(model3,"model3.pkl")

['model3.pkl']

In [ ]:
results={
    "linear regression:":[mae1,rmse1,r2_score1],
    "random forest":[mae2,rmse2,r2_score2],
    "xgboost":[mae3,rmse3,r2_score3]
}
results=pd.DataFrame(results,index=["mae","rmse","r2_score"])
print(results)

          linear regression:  random forest   xgboost
mae                 7.086821       0.262947  0.507575
rmse                9.722490       0.781453  1.007698
r2_score            0.916843       0.999463  0.999107


In [ ]:
print(df[["Usage_kWh", "CO2(tCO2)"]].corr())

           Usage_kWh  CO2(tCO2)
Usage_kWh    1.00000    0.98818
CO2(tCO2)    0.98818    1.00000


In [ ]:
#best model
best=results.T["r2_score"].idxmax()
print(best)

random forest


In [ ]:
joblib.dump(results,"results.pkl")

['results.pkl']